# I-01: Forecasting feature importance (D-39)

Cross-model importance over the forecasters (Track A, hourly; Tower 4 main split):
- **Permutation importance** (model-agnostic, grouped by feature family, **per horizon**) for **RF** (FC-01 tabular) and **LSTM** (FC-02 seq2seq) — the comparable view; tests the hypothesis that AR/CH₄-history matters most at short horizons and future drivers/seasonality grow with horizon.
- **SHAP** (TreeExplainer) on RF — per-feature detail (reuse F-01 pattern).
- **VSN-native** — LSTM+VSN variable-selection gate weights.

Families: `ch4_ar` (CH₄ lags/rolling), `flux_lag` (FCO₂/GPP/Reco, lagged-only), `met` (future met/soil), `planned` (livestock+management), `calendar`, `tower`.

In [1]:
import sys, datetime; sys.path.insert(0, "../../src/models")
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
import forecasting_dl as fdl, shap
from pathlib import Path
RESULTS = Path("../../results"); FIG = RESULTS/"figures"
dev = fdl.get_device(); print("device", dev)
m = fdl.load_matrix()
AR=[c for c in m.columns if c.startswith("ar_")]; FX=fdl.FX; DUM=["is_t2","is_t4","is_t9"]; FEAT=AR+FX+DUM
HSET=[1,24,48]
def is_met(c): return any(x in c for x in ["SWIN","TA_","VPD","PPFD","RN_","WS_","USTAR","SHF","Precip","Soil"])
GROUPS={"ch4_ar":[c for c in AR if "ch4" in c],
        "flux_lag":[c for c in AR if any(x in c for x in ["fc","gpp","reco"])],
        "met":[c for c in FX if is_met(c)],
        "planned":[c for c in FX if any(x in c for x in ["lsu","graze","mgmt"])],
        "calendar":["fx_hs","fx_hc","fx_ds","fx_dc"], "tower":DUM}
print({k:len(v) for k,v in GROUPS.items()})

device cuda


{'ch4_ar': 11, 'flux_lag': 9, 'met': 11, 'planned': 4, 'calendar': 4, 'tower': 3}


## 1  Permutation importance — RF (tabular, per horizon)

In [2]:
def aligned_A(h):
    parts=[]
    for t in [2,4,9]:
        df=m[m.tower==t].set_index("Datetime").sort_index()
        f=df[AR+DUM].copy()
        for c in FX: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["tower"]=t; f["ttime"]=df.index+pd.Timedelta(hours=h)
        parts.append(f)
    return pd.concat(parts)
def perm_rf(h, nrep=3):
    A=aligned_A(h); tr=A[A.ttime.dt.year.between(2018,2021)&A.target.notna()]
    imp=SimpleImputer(strategy="mean"); rf=RandomForestRegressor(300,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(tr[FEAT].values),tr["target"].values)
    te=A[(A.tower==4)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
    X=imp.transform(te[FEAT].values); y=te["target"].values
    base=mean_squared_error(y,rf.predict(X))**0.5; rng=np.random.default_rng(0); out={}
    for g,cols in GROUPS.items():
        idx=[FEAT.index(c) for c in cols]; ds=[]
        for _ in range(nrep):
            Xp=X.copy(); pm=rng.permutation(len(Xp))
            for j in idx: Xp[:,j]=Xp[pm,j]
            ds.append(mean_squared_error(y,rf.predict(Xp))**0.5)
        out[g]=np.mean(ds)-base
    return base,out,(rf,imp,te)
RF_PERM={}; RF_KEEP={}
for h in HSET:
    b,o,keep=perm_rf(h); RF_PERM[h]=o; RF_KEEP[h]=keep
    print(f"RF h={h:2d} base RMSE={b:.1f}  "+" ".join(f"{g}:{v:+.1f}" for g,v in o.items()))

RF h= 1 base RMSE=134.8  ch4_ar:+12.8 flux_lag:+2.1 met:+2.0 planned:+2.6 calendar:+0.1 tower:+0.0


RF h=24 base RMSE=143.0  ch4_ar:+3.9 flux_lag:-0.8 met:+2.7 planned:+5.4 calendar:+0.7 tower:+0.0


RF h=48 base RMSE=143.2  ch4_ar:+1.8 flux_lag:+0.2 met:+3.5 planned:+8.8 calendar:+0.8 tower:+0.0


## 2  Permutation importance — LSTM (seq2seq, per horizon)

In [3]:
W={"A":fdl.build_windows(m,"A")}; Wt=W["A"]; cut=pd.Timestamp("2021-12-31 23:59")
trp=[fdl._subset(Wt[t], pd.DatetimeIndex(Wt[t]["ttime"][:,-1])<=cut) for t in [2,4,9]]
train=fdl._cat(trp); te=fdl._subset(Wt[4], pd.DatetimeIndex(Wt[4]["otime"]).year.isin([2022,2023]))
mu,sd=fdl._standardize(train,[te])
ne,ndf=Wt[4]["enc"].shape[-1],Wt[4]["dec"].shape[-1]
lstm=fdl.build_model("LSTM",168,48,ne,ndf,3); fdl.train_model(lstm,train,dev,epochs=25,ch4_mu=mu,ch4_sd=sd)
ENC=["ch4"]+FX+fdl.FLUX_T; DEC=FX; ei={n:i for i,n in enumerate(ENC)}; di={n:i for i,n in enumerate(DEC)}
DLG={"ch4_ar":(["ch4"],[]), "flux_lag":(fdl.FLUX_T,[]),
     "met":([c for c in FX if is_met(c)],[c for c in FX if is_met(c)]),
     "planned":([c for c in FX if any(x in c for x in ["lsu","graze","mgmt"])],)*2,
     "calendar":(["fx_hs","fx_hc","fx_ds","fx_dc"],)*2}
def rmse_obs(pred,t,h):
    y=t["y"][:,h-1]; ok=np.isfinite(y); return float(np.sqrt(np.mean((pred[ok,h-1]-y[ok])**2)))
def perm_lstm(h,nrep=3):
    base=rmse_obs(fdl.predict(lstm,te,dev,mu,sd),te,h); rng=np.random.default_rng(0); out={}
    for g,(ec,dc) in DLG.items():
        ds=[]
        for _ in range(nrep):
            enc=te["enc"].copy(); dec=te["dec"].copy(); pm=rng.permutation(len(enc))
            for c in ec: enc[:,:,ei[c]]=enc[pm][:,:,ei[c]]
            for c in dc: dec[:,:,di[c]]=dec[pm][:,:,di[c]]
            t2=dict(te); t2["enc"]=enc; t2["dec"]=dec
            ds.append(rmse_obs(fdl.predict(lstm,t2,dev,mu,sd),t2,h))
        out[g]=np.mean(ds)-base
    out["tower"]=0.0
    return out
LSTM_PERM={h:perm_lstm(h) for h in HSET}
for h in HSET: print(f"LSTM h={h:2d}  "+" ".join(f"{g}:{v:+.1f}" for g,v in LSTM_PERM[h].items()))

LSTM h= 1  ch4_ar:+53.0 flux_lag:+8.4 met:+8.7 planned:-8.9 calendar:-3.2 tower:+0.0
LSTM h=24  ch4_ar:-0.0 flux_lag:-0.2 met:+39.5 planned:+7.2 calendar:+8.0 tower:+0.0
LSTM h=48  ch4_ar:+0.3 flux_lag:+0.0 met:+37.6 planned:+7.5 calendar:+14.0 tower:+0.0


## 3  SHAP (RF, h=24) + VSN-native weights

In [4]:
# SHAP on RF h=24
rf,imp,te24=RF_KEEP[24]; Xtr=imp.transform(te24[FEAT].values)
rng=np.random.default_rng(0); sub=rng.choice(len(Xtr),size=min(600,len(Xtr)),replace=False)
sv=shap.TreeExplainer(rf).shap_values(Xtr[sub])
shap_imp=pd.DataFrame({"feature":FEAT,"mean_abs_shap":np.abs(sv).mean(0)}).sort_values("mean_abs_shap",ascending=False)
print("Top 12 SHAP (RF h=24):"); print(shap_imp.head(12).to_string(index=False))
# VSN weights
vsn=fdl.build_model("LSTM_VSN",168,48,ne,ndf,3); fdl.train_model(vsn,train,dev,epochs=25,ch4_mu=mu,ch4_sd=sd); vsn.eval()
with torch.no_grad():
    _=vsn(torch.tensor(te["enc"]).to(dev),torch.tensor(te["dec"]).to(dev),torch.tensor(te["static"]).to(dev))
    w=vsn.last_w.mean(dim=(0,1)).cpu().numpy()
vsn_imp=pd.DataFrame({"feature":ENC,"vsn_weight":w}).sort_values("vsn_weight",ascending=False)
print("\nTop 10 VSN-gate features:"); print(vsn_imp.head(10).to_string(index=False))

Top 12 SHAP (RF h=24):
       feature  mean_abs_shap
   fx_lsu_dens      31.840099
   ar_ch4_lag1       9.084479
  ar_ch4_lag24       7.333922
fx_mgmt_manure       4.942333
   fx_WS_0_0_1       4.081703
  fx_VPD_0_0_1       2.690965
  ar_ch4_lag48       2.254333
   ar_ch4_lag2       2.135187
fx_USTAR_0_0_1       2.072647
       ar_fc_t       1.778438
ar_ch4_rmean24       1.748359
 ar_ch4_lag168       1.724546



Top 10 VSN-gate features:
                          feature  vsn_weight
                              ch4    0.092050
                     fx_VPD_0_0_1    0.062831
                          ar_fc_t    0.056854
                      fx_WS_0_0_1    0.051780
                     fx_SHF_1_1_1    0.048881
                        ar_reco_t    0.047886
                            fx_hc    0.047874
                      fx_RN_1_1_1    0.046931
                         ar_gpp_t    0.046608
fx_Soil Moisture @ 10cm Depth (%)    0.044333


## 4  Figures + summary

In [5]:
# Fig 1: grouped permutation importance vs horizon, RF vs LSTM
fams=list(GROUPS); fig,axes=plt.subplots(1,2,figsize=(13,4.5),sharey=True)
for ax,(title,PERM) in zip(axes,[("RF (tabular)",RF_PERM),("LSTM (seq2seq)",LSTM_PERM)]):
    for g in fams:
        ax.plot(HSET,[PERM[h].get(g,0.0) for h in HSET],"o-",label=g)
    ax.set_title(f"Permutation importance — {title}"); ax.set_xlabel("horizon (h)"); ax.grid(alpha=.3)
axes[0].set_ylabel("ΔRMSE when family shuffled"); axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG/"fc_importance_permutation.png",dpi=130); plt.close(fig)
# Fig 2: SHAP top-15 (RF h24)
fig,ax=plt.subplots(figsize=(8,5)); top=shap_imp.head(15)[::-1]
ax.barh(top["feature"],top["mean_abs_shap"],color="#2ca02c"); ax.set_title("SHAP — RF, h=24 (Tower 4)")
ax.set_xlabel("mean |SHAP|"); fig.tight_layout(); fig.savefig(FIG/"fc_importance_shap_rf.png",dpi=130); plt.close(fig)
# Fig 3: VSN weights top-15
fig,ax=plt.subplots(figsize=(8,5)); topv=vsn_imp.head(15)[::-1]
ax.barh(topv["feature"],topv["vsn_weight"],color="#1f77b4"); ax.axvline(1/len(ENC),ls="--",c="grey",label="uniform")
ax.set_title("LSTM+VSN gate weights (Tower 4 test)"); ax.set_xlabel("mean gate weight"); ax.legend()
fig.tight_layout(); fig.savefig(FIG/"fc_importance_vsn.png",dpi=130); plt.close(fig)
# summary csv
rows=[]
for h in HSET:
    for g in fams:
        rows.append({"method":"perm","model":"RF","horizon":h,"family":g,"value":round(RF_PERM[h].get(g,0.0),3)})
        rows.append({"method":"perm","model":"LSTM","horizon":h,"family":g,"value":round(LSTM_PERM[h].get(g,0.0),3)})
pd.DataFrame(rows).to_csv(RESULTS/"fc_importance_summary.csv",index=False)
shap_imp.to_csv(RESULTS/"fc_importance_shap_rf.csv",index=False)
vsn_imp.to_csv(RESULTS/"fc_importance_vsn.csv",index=False)
print("saved 3 figures + 3 csvs to results/")

saved 3 figures + 3 csvs to results/
